In [2]:
from pathlib import Path
import torch
from rfdetr import RFDETRNano

ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")


In [3]:
DATASET_VERSION = "instance"

DATASET_DIRS = {
    "instance": ROOT / "trashcan_rfdetr",
    "material": ROOT / "trashcan_material_rfdetr"
}

DATASET_DIR = DATASET_DIRS[DATASET_VERSION]

OUTPUT_DIR = ROOT / "outputs" / f"{DATASET_VERSION}_baseline_notebook"

print("Dataset:", DATASET_VERSION)
print("Dataset directory:", DATASET_DIR)

assert (DATASET_DIR / "train" / "_annotations.coco.json").exists()
assert (DATASET_DIR / "valid" / "_annotations.coco.json").exists()

Dataset: instance
Dataset directory: C:\Users\Administrator\COMP9444_RFDETR\trashcan_rfdetr


In [4]:
import json
import shutil
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")
SOURCE = ROOT / "trashcan_raw" / "dataset" / "instance_version"
TARGET = ROOT / "trashcan_instance_rfdetr"

image_files = {}

for p in SOURCE.rglob("*"):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        if p.name in image_files:
            raise ValueError(f"repeated pic: {p.name}")
        image_files[p.name] = p

print("Total images found:", len(image_files))

splits = {
    "train": "instances_train_trashcan.json",
    "valid": "instances_val_trashcan.json"
}

for split, json_name in splits.items():

    matches = list(SOURCE.rglob(json_name))
    assert len(matches) == 1, f": {json_name}"

    with open(matches[0], encoding="utf-8") as f:
        data = json.load(f)

    folder = TARGET / split
    folder.mkdir(parents=True, exist_ok=True)

    for img in data["images"]:
        name = Path(img["file_name"]).name

        assert name in image_files, f"can't find: {name}"

        destination = folder / name

        if not destination.exists():
            shutil.copy2(image_files[name], destination)

        img["file_name"] = name

    with open(
        folder / "_annotations.coco.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f)

    print(
        split,
        "Images:", len(data["images"]),
        "Annotations:", len(data["annotations"]),
        "Categories:", len(data["categories"])
    )

Total images found: 7212
train Images: 6065 Annotations: 9540 Categories: 22
valid Images: 1147 Annotations: 2588 Categories: 22


In [4]:
import time
import random
import numpy as np
import torch

from pathlib import Path
from rfdetr import RFDETRNano


ROOT = Path(r"C:\Users\Administrator\COMP9444_RFDETR")

DATASET_VERSION = "instance"

DATASET_DIR = ROOT / "trashcan_rfdetr"

OUTPUT_DIR = ROOT / "outputs" / "instance_baseline_notebook_10"

EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
SEED = 42


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert DATASET_DIR.exists(), "Dataset directory does not exist."

assert (
    DATASET_DIR / "train" / "_annotations.coco.json"
).exists(), "Training annotations not found."

assert (
    DATASET_DIR / "valid" / "_annotations.coco.json"
).exists(), "Validation annotations not found."


model = RFDETRNano()

start = time.perf_counter()

model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=1,
    lr=LEARNING_RATE,
    num_workers=0,
    device="cuda",
    progress_bar="tqdm",
    compute_val_loss=True,
    output_dir=str(OUTPUT_DIR)
)


elapsed = time.perf_counter() - start

print(f"Training time: {elapsed / 60:.2f} minutes")
print("Output directory:", OUTPUT_DIR)

Val (ema) (Epoch 10/10) — Overall Metrics            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4944 │ 0.6849 │ 0.5512 │ 0.7185 │ 0.6483 │ 0.7411 │ 0.6090 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘


Epoch 9: 100%|██████████| 759/759 [05:28<00:00,  2.31it/s, loss=5.730, loss_cls=0.872, loss_box=0.111, loss_giou=0.254, train/lr=0.0001, val/loss=3.920, val/ema_mAP_50_95=0.494, val/mAP_50_95=0.494, val/F1=0.648, train/loss=3.440, max_mem=5.9 GB, free_mem=379 MB/8.0 GB]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 759/759 [05:35<00:00,  2.26it/s, loss=5.730, loss_cls=0.872, loss_box=0.111, loss_giou=0.254, train/lr=0.0001, val/loss=3.920, val/ema_mAP_50_95=0.494, val/mAP_50_95=0.494, val/F1=0.648, train/loss=3.440, max_mem=5.9 GB, free_mem=379 MB/8.0 GB]
[2026-11-07 14:13:02] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.4944)
Training time: 55.33 minutes
Output directory: C:\Users\Administrator\COMP9444_RFDETR\outputs\instance_baseline_notebook_10
